<a href="https://colab.research.google.com/github/epicariello/zone30/blob/main/Zone30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Questo notebook è stato utilizzato per la realizzazione della tesi L'effetto dell'istituzione della zona 30 km/h sulla mortalità negli incidenti automobilistici**
  

Si carica un dataset ISTAT che contiene gli incidenti con morti e feriti di tutti i comuni d'Italia in serie storica

In [1]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

import pandas as pd
import numpy as np
import re

# File path
file_path = '/content/drive/MyDrive/TesiMagistrale/FontiUsate/Incidenti, morti e feriti - comuni.xlsx'

# Carica dati dalla riga 8 (salta 7 righe), NO header
df = pd.read_excel(file_path, skiprows=7, header=None)

# Crea nomi colonne: Comune + morti/feriti/incidenti per 2001-2024
anni = range(2001, 2025)
colonne = ['Comune']
for anno in anni:
    colonne.extend([f'{anno}_morti', f'{anno}_feriti', f'{anno}_incidenti'])

df.columns = colonne

# 🔧 PULIZIA ROBUSTA per risolvere l'errore '..'
def pulisci_valori(val):
    if pd.isna(val):
        return 0
    val_str = str(val).strip()
    # Rimuovi '..', ',', e non numeri
    if val_str == '..' or val_str == 'nan':
        return 0
    # Rimuovi virgole per numeri italiani
    val_str = val_str.replace(',', '')
    try:
        return int(float(val_str))
    except (ValueError, TypeError):
        return 0

# Applica pulizia a TUTTE le colonne numeriche
colonne_num = [col for col in df.columns if col != 'Comune']
for col in colonne_num:
    df[col] = df[col].apply(pulisci_valori)

# Pulisci 'Comune'
df['Comune'] = df['Comune'].astype(str).str.strip()
df = df[df['Comune'].str.len() > 0].reset_index(drop=True)

print(f"✅ DataFrame 'df' pronto! Shape: {df.shape}")
print("\nPrime righe:")
print(df[['Comune', '2001_morti', '2001_feriti', '2001_incidenti',
          '2024_morti', '2024_feriti', '2024_incidenti']].head())

print("\n📊 Pronto per analisi!")


✅ DataFrame 'df' pronto! Shape: (8578, 73)

Prime righe:
            Comune  2001_morti  2001_feriti  2001_incidenti  2024_morti  \
0            Agliè           0           10               5           0   
1          Airasca           0           48              25           0   
2     Ala di Stura           0            0               0           0   
3  Albiano d'Ivrea           1            6               5           0   
4  Alice Superiore           4            3               3           0   

   2024_feriti  2024_incidenti  
0            5               2  
1            9               5  
2            2               1  
3            9               4  
4            0               0  

📊 Pronto per analisi!


E' necessario un preprocessing poiché alcuni comuni hanno cambiato provincia nel corso del tempo (in particolare è accaduto a Olbia) per questo i dati si trovano in righe diverse, pur essendo, nel complesso, completo

In [2]:
# ============================================================
# PREPROCESSING - Gestione comuni duplicati
# ============================================================

# --- IDENTIFICAZIONE ---
dupes_mask = df['Comune'].duplicated(keep=False)
dupe_names = df.loc[dupes_mask, 'Comune'].unique()
print(f"Comuni con nome duplicato: {len(dupe_names)}")

colonne_num = [col for col in df.columns if col != 'Comune']

# --- CLASSIFICAZIONE ---
# Complementari: dove una riga ha il dato, l'altra ha 0 (mai sovrapposizione)
#                → merge sicuro per somma (es. Olbia cambio provincia)
# Conflitto:     stessa colonna ha valori > 0 su entrambe le righe
#                → comuni omonimi in province diverse, si lasciano separati

complementary = []
conflict = []

for nome in dupe_names:
    rows = df[df['Comune'] == nome]
    has_conflict = False
    for col in colonne_num:
        vals = rows[col].tolist()
        non_zero = [v for v in vals if v != 0]
        if len(non_zero) > 1:
            has_conflict = True
            break
    if has_conflict:
        conflict.append(nome)
    else:
        complementary.append(nome)

print(f"  → Merge per somma (cambio provincia, es. Olbia): {len(complementary)}")
print(f"  → Lasciati separati (comuni omonimi):            {len(conflict)}")
print(f"  → Comuni omonimi non toccati: {conflict}")

# --- MERGE SOLO PER I COMPLEMENTARI ---
idx_to_drop = set()
merged_rows = []

for nome in complementary:
    rows = df[df['Comune'] == nome]
    idx_to_drop.update(rows.index.tolist())
    merged = {'Comune': nome}
    for col in colonne_num:
        merged[col] = rows[col].sum()
    merged_rows.append(merged)

df_merged = pd.DataFrame(merged_rows)

# --- ASSEMBLAGGIO ---
# I comuni in conflitto rimangono intatti nel df originale
df = df.drop(index=list(idx_to_drop))
df = pd.concat([df, df_merged], ignore_index=True)
df = df.sort_values('Comune').reset_index(drop=True)

print(f"\nDataset pulito: {len(df)} righe, {df['Comune'].nunique()} comuni unici")
print(f"(Nota: {len(conflict)} comuni omonimi mantengono righe doppie)")

# --- VERIFICA Bologna e Olbia ---
for nome in ['Bologna', 'Olbia']:
    row = df[df['Comune'] == nome].iloc[0]
    morti = [str(int(row[f'{a}_morti'])) for a in range(2001, 2025)]
    print(f"\n{nome} - Morti 2001→2024:")
    print(', '.join(morti))

print("\n✅ Preprocessing completato! df aggiornato e pronto per l'analisi.")


Comuni con nome duplicato: 294
  → Merge per somma (cambio provincia, es. Olbia): 288
  → Lasciati separati (comuni omonimi):            6
  → Comuni omonimi non toccati: ['Samone', 'Livo', 'Peglio', 'Castro', 'Valverde', 'San Teodoro']

Dataset pulito: 8223 righe, 8215 comuni unici
(Nota: 6 comuni omonimi mantengono righe doppie)

Bologna - Morti 2001→2024:
32, 39, 46, 35, 28, 36, 28, 20, 26, 28, 20, 22, 7, 18, 25, 16, 15, 25, 18, 14, 12, 23, 21, 11

Olbia - Morti 2001→2024:
6, 10, 15, 9, 11, 4, 8, 1, 6, 7, 5, 3, 10, 2, 4, 3, 3, 6, 2, 3, 3, 3, 4, 6

✅ Preprocessing completato! df aggiornato e pronto per l'analisi.


Si carica un dataset dei Comuni dell'Emilia Romagna

In [3]:
import pandas as pd
import numpy as np

# Carica Comuni Emilia-Romagna
df_comuni_emilia_romagna = pd.read_excel('/content/drive/MyDrive/TesiMagistrale/FontiUsate/Comuni_Emilia_Romagna.xlsx')

print("✅ df_comuni_emilia_romagna pronto!")
print(f"Shape: {df_comuni_emilia_romagna.shape}")
print("\nColonne:")
print(df_comuni_emilia_romagna.columns.tolist())
print("\nPrime righe:")
print(df_comuni_emilia_romagna.head())

print("\n📊 Riepilogo:")
print(df_comuni_emilia_romagna['Provincia'].value_counts())
print(f"\nTotale comuni: {len(df_comuni_emilia_romagna)}")


✅ df_comuni_emilia_romagna pronto!
Shape: (330, 4)

Colonne:
['Numero', 'Comune', 'Provincia', 'Popolazione (Censimento 2011)']

Prime righe:
   Numero     Comune      Provincia  Popolazione (Censimento 2011)
0       1   Agazzano       Piacenza                           2070
1       2   Albareto          Parma                           2165
2       3    Albinea  Reggio Emilia                           8755
3       4  Alfonsine        Ravenna                          12245
4       5     Alseno       Piacenza                           4823

📊 Riepilogo:
Provincia
Bologna          55
Modena           47
Piacenza         46
Parma            44
Reggio Emilia    42
Forlì-Cesena     30
Rimini           27
Ferrara          21
Ravenna          18
Name: count, dtype: int64

Totale comuni: 330


Si verifica che tutti i comuni dell'Emilia Romagna siano presenti nel dataset iniziale

In [4]:
# Comuni presenti in df_comuni_emilia_romagna ma ASSENTI in df (incidenti)
comuni_emilia_non_incidenti = set(df_comuni_emilia_romagna['Comune'].str.lower()) - set(df['Comune'].str.lower())

print("🔍 **COMUNI EMILIA-ROMAGNA ASSENTI negli incidenti:**")
print(f"Totale: {len(comuni_emilia_non_incidenti)}")
if len(comuni_emilia_non_incidenti) > 0:
    print(list(comuni_emilia_non_incidenti)[:20])  # Primi 20
    if len(comuni_emilia_non_incidenti) > 20:
        print("... e altri", len(comuni_emilia_non_incidenti)-20)
else:
    print("✅ Tutti i comuni Emilia-Romagna sono presenti!")

print("\n📊 Riepilogo:")
print(f"Comuni Emilia-Romagna totali: {len(df_comuni_emilia_romagna)}")
print(f"Comuni con incidenti: {len(set(df['Comune'].str.lower()) & set(df_comuni_emilia_romagna['Comune'].str.lower()))}")
print(f"Copertura: {100*(1-len(comuni_emilia_non_incidenti)/len(df_comuni_emilia_romagna)):.1f}%")


🔍 **COMUNI EMILIA-ROMAGNA ASSENTI negli incidenti:**
Totale: 0
✅ Tutti i comuni Emilia-Romagna sono presenti!

📊 Riepilogo:
Comuni Emilia-Romagna totali: 330
Comuni con incidenti: 330
Copertura: 100.0%


Si carica un dataset contenente tutti i comuni della Sardegna

In [5]:
import pandas as pd
import numpy as np

# Carica Comuni Sardegna
df_comuni_sardegna = pd.read_excel('/content/drive/MyDrive/TesiMagistrale/FontiUsate/Comuni_Sardegna.xlsx')

print("✅ df_comuni_sardegna pronto!")
print(f"Shape: {df_comuni_sardegna.shape}")
print("\nColonne:")
print(df_comuni_sardegna.columns.tolist())
print("\nPrime righe:")
print(df_comuni_sardegna.head())

print("\n📊 Riepilogo:")
print(df_comuni_sardegna['Provincia'].value_counts())
print(f"\nTotale comuni: {len(df_comuni_sardegna)}")


✅ df_comuni_sardegna pronto!
Shape: (377, 5)

Colonne:
['Comune', 'Provincia', 'Popolazione (2021)', 'Altitudine (m)', 'Superficie (km²)']

Prime righe:
          Comune                  Provincia  Popolazione (2021)  \
0      Abbasanta                   Oristano                2579   
1         Aggius  Gallura Nord-Est Sardegna                1409   
2       Aglientu  Gallura Nord-Est Sardegna                1154   
3   Aidomaggiore                   Oristano                 398   
4  Alà dei Sardi  Gallura Nord-Est Sardegna                1764   

   Altitudine (m)  Superficie (km²)  
0             315             39.85  
1             514             86.31  
2             420            148.19  
3             250             41.21  
4             663            197.99  

📊 Riepilogo:
Provincia
Oristano                     87
Cagliari                     70
Sassari                      66
Nuoro                        53
Medio Campidano              28
Gallura Nord-Est Sardegna    26


Si verifica che tutti i Comuni della Sardegna siano nel dataset degli incidenti

In [6]:
# Verifica comuni SARDAGNA nel df incidenti
comuni_sardegna = set(df_comuni_sardegna['Comune'].str.lower())
comuni_incidenti = set(df['Comune'].str.lower())

comuni_sard_non_incidenti = comuni_sardegna - comuni_incidenti

print("🔍 **COMUNI SARDAGNA ASSENTI negli incidenti:**")
print(f"Totale: {len(comuni_sard_non_incidenti)}")
if len(comuni_sard_non_incidenti) > 0:
    print("\nPrimi 20:")
    for i, comune in enumerate(sorted(comuni_sard_non_incidenti)[:20]):
        print(f"  {i+1}. {comune.title()}")
    if len(comuni_sard_non_incidenti) > 20:
        print(f"\n... e altri {len(comuni_sard_non_incidenti)-20}")
else:
    print("✅ Tutti i comuni Sardegna SONO presenti!")

print(f"\n📊 Copertura:")
print(f"Comuni Sardegna totali: {len(df_comuni_sardegna):,}")
print(f"Comuni con incidenti: {len(comuni_sardegna & comuni_incidenti):,}")
print(f"Copertura: {100*(len(comuni_sardegna & comuni_incidenti)/len(df_comuni_sardegna)):.1f}%")


🔍 **COMUNI SARDAGNA ASSENTI negli incidenti:**
Totale: 0
✅ Tutti i comuni Sardegna SONO presenti!

📊 Copertura:
Comuni Sardegna totali: 377
Comuni con incidenti: 377
Copertura: 100.0%


In [8]:

import pandas as pd

# Carica il CSV della popolazione per comune per anno
df_pop = pd.read_csv(
    '/content/drive/MyDrive/TesiMagistrale/FontiUsate/popolazione_comuni_per_anno.csv',
    sep=';',
    decimal=',',
    encoding='latin-1'
)

# Rinomina per chiarezza
df_pop = df_pop.rename(columns={'codice': 'Codice', 'comune': 'Comune'})

print("✅ df_pop pronto!")
print(f"Shape: {df_pop.shape}")
print("\nColonne:")
print(df_pop.columns.tolist())
print("\nPrime righe:")
print(df_pop.head())

# -------------------------------------------------------
# Verifica: tutti i comuni di df sono presenti in df_pop?
# -------------------------------------------------------
comuni_df = set(df['Comune'].str.strip().str.lower())
comuni_pop = set(df_pop['Comune'].str.strip().str.lower())

assenti_in_pop = comuni_df - comuni_pop

print("\n🔍 Comuni presenti in df (incidenti) ma ASSENTI in df_pop (popolazione):")
print(f"Totale: {len(assenti_in_pop)}")
if assenti_in_pop:
    for c in sorted(assenti_in_pop):
        print(f"  - {c.title()}")
else:
    print("✅ Tutti i comuni di df sono presenti in df_pop!")

print(f"\n📊 Riepilogo copertura:")
print(f"Comuni in df (incidenti):   {len(comuni_df):,}")
print(f"Comuni in df_pop:            {len(comuni_pop):,}")
print(f"Comuni coperti:              {len(comuni_df & comuni_pop):,}")
print(f"Copertura: {100 * len(comuni_df & comuni_pop) / len(comuni_df):.1f}%")


✅ df_pop pronto!
Shape: (7991, 26)

Colonne:
['Codice', 'Comune', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

Prime righe:
   Codice           Comune    2002    2003    2004    2005    2006    2007  \
0    1001            Agliè  2557.0  2538.0  2588.0  2679.0  2674.0  2662.0   
1    1002          Airasca  3543.0  3544.0  3605.0  3648.0  3642.0  3666.0   
2    1003     Ala di Stura   480.0   468.0   461.0   458.0   458.0   467.0   
3    1004  Albiano d'Ivrea  1687.0  1673.0  1702.0  1696.0  1687.0  1661.0   
4    1006           Almese  5648.0  5679.0  5805.0  5863.0  6134.0  6157.0   

     2008    2009  ...    2016    2017    2018    2019    2020    2021  \
0  2658.0  2638.0  ...  2644.0  2661.0  2658.0  2634.0  2621.0  2545.0   
1  3775.0  3794.0  ...  3761.0  3726.0  3671.0  3613.0  3598.0  3633.0   
2   480.0   477.0  ...   464.0   460.

In [10]:
import pandas as pd
import numpy as np

# -------------------------------------------------------
# Costruzione dataset definitivo
# -------------------------------------------------------

# Chiave di join case-insensitive (colonna temporanea)
df_tmp = df.copy()
df_tmp['_key'] = df_tmp['Comune'].str.strip().str.lower()

df_pop_tmp = df_pop.copy()
df_pop_tmp['_key'] = df_pop_tmp['Comune'].str.strip().str.lower()

# Inner join → solo comuni presenti in entrambi
df_final = df_tmp.merge(df_pop_tmp, on='_key', suffixes=('', '_pop'))

# Rimuovi colonne ausiliarie
df_final = df_final.drop(columns=['_key', 'Comune_pop'])

# -------------------------------------------------------
# Calcolo tassi per gli anni sovrapposti: 2002–2024
# (df_pop copre 2002-2025, df copre 2001-2024)
# -------------------------------------------------------
anni_comuni = range(2002, 2025)

for anno in anni_comuni:
    pop = df_final[str(anno)].replace(0, np.nan)
    df_final[f'{anno}_tasso_mortalita']  = (df_final[f'{anno}_morti']  / pop * 1000).round(4)
    df_final[f'{anno}_tasso_ferimento']  = (df_final[f'{anno}_feriti'] / pop * 1000).round(4)

# Rimuovi le colonne grezze di popolazione (2002-2025)
pop_year_cols = [str(a) for a in range(2002, 2026) if str(a) in df_final.columns]
df_final = df_final.drop(columns=pop_year_cols)

# Rimuovi anno 2001 (no dati popolazione → niente tassi) e 2025 (no dati incidenti)
cols_2001 = [c for c in df_final.columns if c.startswith('2001_')]
cols_2025 = [c for c in df_final.columns if c.startswith('2025_') or c == '2025']
df_final = df_final.drop(columns=cols_2001 + cols_2025, errors='ignore')

# Rimuovi Codice
df_final = df_final.drop(columns=['Codice'], errors='ignore')

df_final = df_final.reset_index(drop=True)

print("✅ df_final pronto!")
print(f"Shape: {df_final.shape}")
print(f"Comuni: {len(df_final):,}")
print(f"Anni coperti: 2002–2024")

print("\nColonne (prime e ultime):")
cols = df_final.columns.tolist()
print(cols[:4], "...", cols[-6:])

print("\nEsempio — Olbia:")
esempio = df_final[df_final['Comune'] == 'Olbia']
if not esempio.empty:
    for anno in [2002, 2010, 2024]:
        row = esempio.iloc[0]
        print(f"  {anno}: morti={row[f'{anno}_morti']}, feriti={row[f'{anno}_feriti']}, "
              f"tasso_mortalita={row[f'{anno}_tasso_mortalita']}, "
              f"tasso_ferimento={row[f'{anno}_tasso_ferimento']}")

print(f"\n📊 Comuni rimossi (assenti in df_pop): "
      f"{len(df) - len(df_final)} su {len(df)}")

✅ df_final pronto!
Shape: (7993, 116)
Comuni: 7,993
Anni coperti: 2002–2024

Colonne (prime e ultime):
['Comune', '2002_morti', '2002_feriti', '2002_incidenti'] ... ['2022_tasso_mortalita', '2022_tasso_ferimento', '2023_tasso_mortalita', '2023_tasso_ferimento', '2024_tasso_mortalita', '2024_tasso_ferimento']

Esempio — Olbia:
  2002: morti=10, feriti=521, tasso_mortalita=0.2204, tasso_ferimento=11.4841
  2010: morti=7, feriti=466, tasso_mortalita=0.1315, tasso_ferimento=8.7554
  2024: morti=6, feriti=381, tasso_mortalita=0.0976, tasso_ferimento=6.197

📊 Comuni rimossi (assenti in df_pop): 230 su 8223


In [16]:
print(df_final .columns.tolist())

['Comune', '2002_morti', '2002_feriti', '2002_incidenti', '2003_morti', '2003_feriti', '2003_incidenti', '2004_morti', '2004_feriti', '2004_incidenti', '2005_morti', '2005_feriti', '2005_incidenti', '2006_morti', '2006_feriti', '2006_incidenti', '2007_morti', '2007_feriti', '2007_incidenti', '2008_morti', '2008_feriti', '2008_incidenti', '2009_morti', '2009_feriti', '2009_incidenti', '2010_morti', '2010_feriti', '2010_incidenti', '2011_morti', '2011_feriti', '2011_incidenti', '2012_morti', '2012_feriti', '2012_incidenti', '2013_morti', '2013_feriti', '2013_incidenti', '2014_morti', '2014_feriti', '2014_incidenti', '2015_morti', '2015_feriti', '2015_incidenti', '2016_morti', '2016_feriti', '2016_incidenti', '2017_morti', '2017_feriti', '2017_incidenti', '2018_morti', '2018_feriti', '2018_incidenti', '2019_morti', '2019_feriti', '2019_incidenti', '2020_morti', '2020_feriti', '2020_incidenti', '2021_morti', '2021_feriti', '2021_incidenti', '2022_morti', '2022_feriti', '2022_incidenti', '2